# **P3: Acquisition & Integration**



---
**DSA 405 · Fall 2026 · Project Milestone 3**


This notebook acquires the Austin Animal Center Outcomes dataset, documents the acquisition process and its ethical/legal considerations, integrates it with the cleaned Intakes dataset from P2, and verifies the resulting join.

The analysis uses animal IDs and chronological event order to match intake records with outcome records. Because the source data can contain multiple records for an animal, the matching rule and its limitations are documented before the merge.

### 1. Acquisition Code

The Outcomes dataset is published by the City of Austin Open Data Portal. The public CSV export endpoint is used below. The raw response is saved under data/raw/ so that subsequent notebook runs use the cached copy instead of downloading the same resource again.

In [1]:
from pathlib import Path
from io import BytesIO
import time
import requests
import pandas as pd

outcomes_url = "https://data.austintexas.gov/api/v3/views/9t4d-g238/export.csv"

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

outcomes_raw_file = raw_dir / "austin_animal_outcomes.csv"

expected_outcome_columns = [
    "Animal ID",
    "Date of Birth",
    "Name",
    "DateTime",
    "MonthYear",
    "Outcome Type",
    "Outcome Subtype",
    "Animal Type",
    "Sex upon Outcome",
    "Age upon Outcome",
    "Breed",
    "Color",
]

downloaded_content = None

if outcomes_raw_file.exists():
    print("Using cached Outcomes file:", outcomes_raw_file)
    outcomes = pd.read_csv(outcomes_raw_file)

else:
    headers = {
        "User-Agent": "DSA405-StudentProject/1.0",
        "Accept": "text/csv",
    }

    # Respect the one-second crawl-delay identified in robots.txt.
    time.sleep(1)

    response = requests.get(
        outcomes_url,
        headers=headers,
        timeout=60,
    )

    # Fail loudly for unsuccessful HTTP responses.
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()

    if "csv" not in content_type and "text/plain" not in content_type:
        raise ValueError(
            f"Expected a CSV response, but received Content-Type: {content_type}"
        )

    if not response.content:
        raise ValueError("The CSV response was empty.")

    # Inspect response headers for any rate-limit information.
    rate_limit_headers = {
        key: value
        for key, value in response.headers.items()
        if any(term in key.lower() for term in ["rate", "limit", "retry-after"])
    }

    if rate_limit_headers:
        print("Rate-limit headers:", rate_limit_headers)
    else:
        print("No explicit rate-limit headers returned.")

    # Parse the downloaded CSV.
    outcomes = pd.read_csv(BytesIO(response.content))
    downloaded_content = response.content

# Validate the dataset whether it was downloaded or loaded from the cache.
missing_columns = set(expected_outcome_columns) - set(outcomes.columns)

if missing_columns:
    raise ValueError(
        f"Expected columns missing from CSV: {missing_columns}"
    )

if outcomes.empty:
    raise ValueError("The Outcomes dataset contains no records.")

if outcomes["Animal ID"].isna().any():
    raise ValueError("The Outcomes dataset contains missing Animal IDs.")

# Save the downloaded raw response only after validation succeeds.
if downloaded_content is not None:
    outcomes_raw_file.write_bytes(downloaded_content)
    print("Downloaded and cached raw Outcomes data:", outcomes_raw_file)

print("Outcomes dataset shape:", outcomes.shape)

Using cached Outcomes file: data/raw/austin_animal_outcomes.csv
Outcomes dataset shape: (173775, 12)


In [2]:
# Demonstrate that an HTTP 403 response is caught and reported cleanly.
# This locally reproduces the HTTP error encountered with the API query endpoint;
# it does not make another network request.

try:
    simulated_response = requests.Response()
    simulated_response.status_code = 403
    simulated_response.url = (
        "https://data.austintexas.gov/resource/9t4d-g238.json"
    )
    simulated_response.reason = "Forbidden"
    simulated_response.raise_for_status()
except requests.exceptions.HTTPError as error:
    print("Caught expected HTTP failure:", error)

Caught expected HTTP failure: 403 Client Error: Forbidden for url: https://data.austintexas.gov/resource/9t4d-g238.json


For this part of the project, I downloaded the Animal Outcomes dataset from the City of Austin Open Data Portal using its public CSV export endpoint. Initially, I tried using the API query endpoint, but it returned an HTTP 403 authentication error. Instead of trying to get around that restriction, I switched to the public CSV export endpoint, which successfully provided the same dataset without requiring authentication.

In my acquisition code, I included a User-Agent identifying this as a DSA 405 student project, a 60-second timeout, and a one-second delay before making the request to respect the crawl-delay listed in the website's robots.txt file. I also added checks to make sure the request was successful, the response contained CSV data, and the dataset had the expected columns and no missing Animal IDs. The code also checks the response headers for rate-limit information.

I saved the downloaded data in data/raw/ so that I wouldn't have to download it again every time I ran the notebook. If the file is already there, the code reads from the saved copy instead and prints that the data is already cached to ensure that it works. The dataset downloaded successfully with 173,775 rows and 12 columns. There were no missing values in the Animal ID column and 156,237 unique Animal IDs.

I also included a failure demonstration using the HTTP 403 error I encountered earlier. I reproduced the error locally and showed that the code could catch it using try/except instead of allowing it to interrupt the notebook. The CSV export did not return any explicit rate-limit headers, so I used the one-second crawl-delay from robots.txt as my request interval. Since the export returned the entire dataset in one response, pagination was not needed.

Now, let's inspect the outcomes data

In [3]:
print("Dataset shape:", outcomes.shape)
print("\nColumn names:")
print(outcomes.columns.tolist())

print("\nData types:")
display(outcomes.dtypes.to_frame("dtype"))

print("\nFirst five rows:")
display(outcomes.head())

print("\nMissing values by column:")
missing_summary = pd.DataFrame({
    "missing_count": outcomes.isna().sum(),
    "missing_rate": outcomes.isna().mean()
})
display(missing_summary)

print("\nUnique Animal IDs:", outcomes["Animal ID"].nunique())
print("Missing Animal IDs:", outcomes["Animal ID"].isna().sum())

Dataset shape: (173775, 12)

Column names:
['Animal ID', 'Date of Birth', 'Name', 'DateTime', 'MonthYear', 'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome', 'Age upon Outcome', 'Breed', 'Color']

Data types:


,dtype
Animal ID,object
Date of Birth,object
Name,object
DateTime,object
MonthYear,object
Outcome Type,object
Outcome Subtype,object
Animal Type,object
Sex upon Outcome,object
Age upon Outcome,object



First five rows:


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black



Missing values by column:


,missing_count,missing_rate
Animal ID,0,0.000000
Date of Birth,0,0.000000
Name,49784,0.286485
DateTime,0,0.000000
MonthYear,0,0.000000
Outcome Type,46,0.000265
Outcome Subtype,94115,0.541591
Animal Type,0,0.000000
Sex upon Outcome,1,0.000006
Age upon Outcome,9,0.000052



Unique Animal IDs: 156237
Missing Animal IDs: 0


### 2. Ethics and Legality in Practice

The Animal Outcomes dataset comes from the City of Austin Open Data Portal. According to the portal's terms of use, its data is available free of charge and without restriction. The data is also identified as public domain, with a request to provide attribution to both the City of Austin and the department responsible for the data. My project uses publicly available animal shelter records and does not involve accessing private information, bypassing a login or paywall, or attempting to identify or contact individuals. I will also avoid combining datasets for the purpose of identifying individuals, as prohibited by the portal's terms.

I checked the portal's robots.txt file and found that it specifies a one-second crawl delay for all user agents. I followed this guidance by including a one-second delay before making my download request. My code also uses a User-Agent identifying the request as part of a DSA 405 student project. The acquisition uses the portal's public CSV export endpoint and does not require authentication or circumvent any access restrictions.

There was one change from my original A8 plan. I initially planned to use the dataset's API query endpoint, and it worked during my A8 feasibility test. However, when I tried it again for P3, it returned a 403 authentication error. This may have happened because the endpoint's authentication requirements or access settings changed between the two attempts; the error message indicated that authentication or an application token was now required. Rather than trying to bypass that requirement, I switched to the portal's public CSV export endpoint. This allowed me to download the same Animal Outcomes dataset without authentication. The source dataset itself did not change; only the method used to acquire it changed.

The CSV export returned the complete dataset in one response, so pagination was not necessary. I also checked the response headers for rate-limit information, but the server did not return any explicit rate-limit headers. I therefore followed the one-second crawl delay specified in robots.txt.

### 3. Join Design

Before we get into the join design, let's load the cleaned intakes data that we prepared in P2

In [4]:
intakes_clean_url = (
    "https://raw.githubusercontent.com/"
    "SoumyaPannala/dsa405-project/main/"
    "data/raw/austin_animal_intakes_clean.csv.gz"
)

intakes_clean = pd.read_csv(
    intakes_clean_url,
    compression="gzip"
)

print("Intakes dataset shape:", intakes_clean.shape)
print("\nIntakes columns:")
print(intakes_clean.columns.tolist())
print("\nFirst five rows:")
display(intakes_clean.head())

Intakes dataset shape: (173812, 13)

Intakes columns:
[':id', 'animal_id', 'name', 'datetime', 'found_location', 'intake_type', 'intake_condition', 'animal_type', 'sex_upon_intake', 'age_upon_intake', 'breed', 'color', 'age_years']

First five rows:


,:id,animal_id,name,datetime,found_location,intake_type,intake_condition,animal_type,sex_upon_intake,age_upon_intake,breed,color,age_years
0,row-5scy-j6d5-2y33,A521520,Nina,2013-10-01T07:51:00.000,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan,7.000000
1,row-kpt7~mbcv~uxgj,A664235,NaN,2013-10-01T08:33:00.000,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White,0.019165
2,row-nrmv~7qgv~tqjb,A664236,NaN,2013-10-01T08:33:00.000,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White,0.019165
3,row-b36g-3dxp.q3k3,A664237,NaN,2013-10-01T08:33:00.000,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White,0.019165
4,row-gwpa~iepp-4323,A664233,Stevie,2013-10-01T08:53:00.000,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,3 years,Pit Bull Mix,Blue/White,3.000000


Now we'll start preparing the data for the join

In [5]:
intakes_join = intakes_clean.copy()
outcomes_join = outcomes.copy()

# Standardize the shared animal ID field.
outcomes_join = outcomes_join.rename(
    columns={"Animal ID": "animal_id"}
)

intakes_join["animal_id"] = (
    intakes_join["animal_id"].astype("string").str.strip()
)
outcomes_join["animal_id"] = (
    outcomes_join["animal_id"].astype("string").str.strip()
)

print("Intakes records:", len(intakes_join))
print("Outcomes records:", len(outcomes_join))
print("Unique Intakes animal IDs:", intakes_join["animal_id"].nunique())
print("Unique Outcomes animal IDs:", outcomes_join["animal_id"].nunique())

Intakes records: 173812
Outcomes records: 173775
Unique Intakes animal IDs: 156287
Unique Outcomes animal IDs: 156237


In [6]:
intake_ids = set(intakes_join["animal_id"].dropna().unique())
outcome_ids = set(outcomes_join["animal_id"].dropna().unique())

shared_ids = intake_ids & outcome_ids
intake_only_ids = intake_ids - outcome_ids
outcome_only_ids = outcome_ids - intake_ids

print("Animals appearing in both sources:", len(shared_ids))
print("Animals only in Intakes:", len(intake_only_ids))
print("Animals only in Outcomes:", len(outcome_only_ids))

Animals appearing in both sources: 155423
Animals only in Intakes: 864
Animals only in Outcomes: 814


In [7]:
intake_counts = intakes_join["animal_id"].value_counts()
outcome_counts = outcomes_join["animal_id"].value_counts()

record_comparison = pd.concat(
    [
        intake_counts.rename("intake_records"),
        outcome_counts.rename("outcome_records"),
    ],
    axis=1
).fillna(0).astype(int)

matching_record_counts = (
    record_comparison["intake_records"]
    == record_comparison["outcome_records"]
)
different_record_counts = ~matching_record_counts

print("Animals with matching record counts:", matching_record_counts.sum())
print("Animals with different record counts:", different_record_counts.sum())

print("\nExamples with different record counts:")
display(record_comparison.loc[different_record_counts].head(10))

Animals with matching record counts: 155215
Animals with different record counts: 1886

Examples with different record counts:


,intake_records,outcome_records
animal_id,,
A705625,12,11
A812875,4,3
A776757,4,3
A867245,4,3
A661049,4,5
A664228,4,5
A913619,4,3
A871827,3,2
A901857,3,2


We now have looked at both datasets and can design our join.

| Design element | Decision |
|---|---|
| Join keys | `animal_id` and `event_number` |
| Grain of the left table | One row per intake event for an animal |
| Grain of the right table | One row per outcome event for an animal |
| Relationship | One-to-one after assigning chronological event numbers within each animal |
| Expected row count after the join | 172,850 matched event pairs |
| Join type and why | Inner join, to retain only intake and outcome events that can be matched |
| What happens to unmatched rows | Unmatched intake and outcome records are excluded from the joined dataset |

The original animal_id field is not unique in either dataset. To avoid a many-to-many join that could multiply records, I planned to sort each dataset chronologically and assign an event_number within each animal. The merge therefore uses both animal_id and event_number.

The expected row count is 172,850, calculated as the sum of the smaller record count for each animal across the two datasets. The validate="one_to_one" argument checks that each event pair is unique and protects against an unintended fan-out join.

The inner join excludes unmatched records, so it can remove observations that do not have a corresponding event in the other dataset. I report those unmatched records separately.

Additionally, my use of validate="one_to_one" guards against fan-out by preventing duplicate event matches from multiplying rows. It does not prevent the row loss associated with an inner join, so I separately calculate and report the unmatched intake and outcome records.

Now, let's make sure the dates are parsed and assign event numbers before we join the two datasets.

In [8]:
# Intakes dates parse successfully as UTC timestamps.
intakes_join["datetime"] = pd.to_datetime(
    intakes_join["datetime"],
    errors="coerce",
    utc=True
)

# Outcomes contains timestamp strings both with and without timezone offsets.
# Identify timestamps ending in an offset such as -05:00.
has_timezone = outcomes_join["DateTime"].astype("string").str.strip().str.contains(
    r"[+-]\d{2}:\d{2}$",
    na=False
)

print("Timestamps with timezone offsets:", has_timezone.sum())
print("Timestamps without timezone offsets:", (~has_timezone).sum())

parsed_dates = pd.Series(
    pd.NaT,
    index=outcomes_join.index,
    dtype="datetime64[ns, UTC]"
)

# Parse timestamps that include timezone offsets.
parsed_dates.loc[has_timezone] = pd.to_datetime(
    outcomes_join.loc[has_timezone, "DateTime"],
    format="%Y-%m-%dT%H:%M:%S%z",
    utc=True
)

# Parse timestamps without timezone offsets.
parsed_dates.loc[~has_timezone] = pd.to_datetime(
    outcomes_join.loc[~has_timezone, "DateTime"],
    format="%Y-%m-%dT%H:%M:%S",
    utc=True
)

outcomes_join["DateTime"] = parsed_dates

print("Parsed Intakes dates:", intakes_join["datetime"].notna().sum())
print("Total Intakes records:", len(intakes_join))
print("Parsed Outcomes dates:", outcomes_join["DateTime"].notna().sum())
print("Total Outcomes records:", len(outcomes_join))

Timestamps with timezone offsets: 3584
Timestamps without timezone offsets: 170191
Parsed Intakes dates: 173812
Total Intakes records: 173812
Parsed Outcomes dates: 173775
Total Outcomes records: 173775


In [9]:
# Sort each dataset chronologically within animal ID.
intakes_join = intakes_join.sort_values(
    ["animal_id", "datetime"]
).copy()

outcomes_join = outcomes_join.sort_values(
    ["animal_id", "DateTime"]
).copy()

# Number events within each animal, starting at 1.
intakes_join["event_number"] = (
    intakes_join.groupby("animal_id").cumcount() + 1
)

outcomes_join["event_number"] = (
    outcomes_join.groupby("animal_id").cumcount() + 1
)

Now for the merge.

In [10]:
animal_join = intakes_join.merge(
    outcomes_join,
    on=["animal_id", "event_number"],
    how="inner",
    suffixes=("_intake", "_outcome"),
    validate="one_to_one"
)

print("Joined records:", len(animal_join))
print("Unique animals:", animal_join["animal_id"].nunique())
print("Joined columns:", len(animal_join.columns))

display(animal_join.head())

Joined records: 172850
Unique animals: 155423
Joined columns: 25


,:id,animal_id,name,datetime,found_location,intake_type,intake_condition,animal_type,sex_upon_intake,age_upon_intake,...,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,row-6cme~rzjh-6je4,A006100,Scamp,2014-03-07 14:26:00+00:00,8700 Research in Austin (TX),Public Assist,Normal,Dog,Neutered Male,6 years,...,Scamp,2014-03-08 17:10:00+00:00,03-2014,Return to Owner,NaN,Dog,Neutered Male,6 years,Spinone Italiano Mix,Yellow/White
1,row-c6ws_7x4p-qb2j,A006100,Scamp,2014-12-19 10:21:00+00:00,8700 Research Blvd in Austin (TX),Public Assist,Normal,Dog,Neutered Male,7 years,...,Scamp,2014-12-20 16:35:00+00:00,12-2014,Return to Owner,NaN,Dog,Neutered Male,7 years,Spinone Italiano Mix,Yellow/White
2,row-22nu.wj4k_yifn,A006100,Scamp,2017-12-07 14:07:00+00:00,Colony Creek And Hunters Trace in Austin (TX),Stray,Normal,Dog,Neutered Male,10 years,...,Scamp,2017-12-07 05:00:00+00:00,12-2017,Return to Owner,NaN,Dog,Neutered Male,10 years,Spinone Italiano Mix,Yellow/White
3,row-ktv7_2cab-2hfn,A047759,Oreo,2014-04-02 15:55:00+00:00,Austin (TX),Owner Surrender,Normal,Dog,Neutered Male,10 years,...,Oreo,2014-04-07 15:12:00+00:00,04-2014,Transfer,Partner,Dog,Neutered Male,10 years,Dachshund,Tricolor
4,row-kfqw.y7yf_a7rj,A134067,Bandit,2013-11-16 09:02:00+00:00,12034 Research Blvd in Austin (TX),Public Assist,Injured,Dog,Neutered Male,16 years,...,Bandit,2013-11-16 11:54:00+00:00,11-2013,Return to Owner,NaN,Dog,Neutered Male,16 years,Shetland Sheepdog,Brown/White


Now we'll examine date differences and prepare the analysis dataset.

In [11]:
# First compare exact timestamps to identify records that need closer inspection.
animal_join["exact_days_to_outcome"] = (
    animal_join["DateTime"] - animal_join["datetime"]
).dt.total_seconds() / 86400

print(
    "Negative exact timestamp intervals:",
    (animal_join["exact_days_to_outcome"] < 0).sum()
)
display(animal_join["exact_days_to_outcome"].describe())

Negative exact timestamp intervals: 780


,exact_days_to_outcome
count,172850.000000
mean,20.695291
std,51.917214
min,-2745.900000
25%,1.996528
50%,5.993056
75%,19.869618
max,1912.938194


In [12]:
# Compare calendar dates to avoid treating same-day time/timezone differences
# as an outcome occurring before intake.
animal_join["days_to_outcome"] = (
    animal_join["DateTime"].dt.date -
    animal_join["datetime"].dt.date
).apply(lambda difference: difference.days)

same_day_pairs = (animal_join["days_to_outcome"] == 0).sum()
reversed_date_pairs = (animal_join["days_to_outcome"] < 0).sum()
later_outcome_pairs = (animal_join["days_to_outcome"] > 0).sum()

print("Same-day pairs:", same_day_pairs)
print("Outcome dates before intake dates:", reversed_date_pairs)
print("Outcome dates after intake dates:", later_outcome_pairs)

Same-day pairs: 22353
Outcome dates before intake dates: 139
Outcome dates after intake dates: 150358


In [13]:
animal_analysis = animal_join.loc[
    animal_join["days_to_outcome"] >= 0
].copy()

print("Original joined records:", len(animal_join))
print("Excluded records:", len(animal_join) - len(animal_analysis))
print("Valid analysis records:", len(animal_analysis))
print("Unique animals in analysis:", animal_analysis["animal_id"].nunique())

display(animal_analysis["days_to_outcome"].describe())

Original joined records: 172850
Excluded records: 139
Valid analysis records: 172711
Unique animals in analysis: 155313


,days_to_outcome
count,172711.000000
mean,20.965161
std,48.335212
min,0.000000
25%,2.000000
50%,6.000000
75%,20.000000
max,1913.000000


The initial exact-timestamp comparison identified 780 negative intervals. Further inspection showed that many were due to time-of-day or timezone differences on the same calendar date. After comparing calendar dates, 139 pairs remained where the outcome date appeared before the intake date. I excluded those pairs from the time-to-outcome analysis, leaving 172,711 valid pairs.

### 4. Verification Suite

Now, let's do the verification to ensure that the merge went as expected.

In [14]:
expected_pairs = record_comparison.min(axis=1).sum()

# Protects against a fan-out join that multiplies event rows.
assert len(animal_join) == expected_pairs, (
    "Unexpected number of joined records"
)

# Protects against duplicate event keys in the joined dataset.
assert not animal_join.duplicated(
    subset=["animal_id", "event_number"]
).any(), "Duplicate event matches found"

# Protects against missing or unparsed intake dates entering the join.
assert animal_join["datetime"].notna().all(), (
    "Missing intake dates in joined data"
)

# Protects against missing or unparsed outcome dates entering the join.
assert animal_join["DateTime"].notna().all(), (
    "Missing outcome dates in joined data"
)

# Protects against invalid negative calendar-day intervals in the analysis data.
assert (animal_analysis["days_to_outcome"] >= 0).all(), (
    "Negative time-to-outcome values remain"
)

print("All join verification checks passed.")
print("Expected matched pairs:", expected_pairs)
print("Actual matched pairs:", len(animal_join))
print(
    "Duplicate event matches:",
    animal_join.duplicated(
        subset=["animal_id", "event_number"]
    ).sum()
)

All join verification checks passed.
Expected matched pairs: 172850
Actual matched pairs: 172850
Duplicate event matches: 0


Now, we'll create a join summary

In [15]:
join_summary = pd.DataFrame({
    "Measure": [
        "Intake records",
        "Outcome records",
        "Unique animals in Intakes",
        "Unique animals in Outcomes",
        "Animals appearing in both sources",
        "Animals only in Intakes",
        "Animals only in Outcomes",
        "Matched event pairs",
        "Unmatched intake records",
        "Unmatched outcome records",
        "Valid pairs retained for analysis",
        "Pairs excluded for reversed dates",
    ],
    "Count": [
        len(intakes_join),
        len(outcomes_join),
        intakes_join["animal_id"].nunique(),
        outcomes_join["animal_id"].nunique(),
        len(shared_ids),
        len(intake_only_ids),
        len(outcome_only_ids),
        len(animal_join),
        len(intakes_join) - len(animal_join),
        len(outcomes_join) - len(animal_join),
        len(animal_analysis),
        len(animal_join) - len(animal_analysis),
    ]
})

display(join_summary)

,Measure,Count
0,Intake records,173812
1,Outcome records,173775
2,Unique animals in Intakes,156287
3,Unique animals in Outcomes,156237
4,Animals appearing in both sources,155423
5,Animals only in Intakes,864
6,Animals only in Outcomes,814
7,Matched event pairs,172850
8,Unmatched intake records,962
9,Unmatched outcome records,925


### Data Integration and Join Strategy

I integrated the cleaned Animal Intakes dataset with the Animal Outcomes dataset using the shared `animal_id` field. Before joining, I standardized the ID columns as strings and removed leading and trailing whitespace.

Both datasets contained multiple records for some animals, so a direct merge on `animal_id` could have produced a many-to-many join and artificially multiplied records. Instead, I sorted each dataset chronologically and assigned an event number to each record within its animal ID. I then joined the datasets using both `animal_id` and `event_number`. This paired records in chronological order and prevented duplicate event matches.

The Intakes dataset contained 173,812 records representing 156,287 unique animals. The Outcomes dataset contained 173,775 records representing 156,237 unique animals. There were 155,423 animal IDs appearing in both datasets, while 864 appeared only in Intakes and 814 appeared only in Outcomes.

The event-level join produced 172,850 matched pairs, with 962 unmatched intake records and 925 unmatched outcome records. I checked the dates for each pair and found 139 cases where the outcome date preceded the intake date. I excluded these records from the time-to-outcome analysis, leaving 172,711 records representing 155,313 unique animals.

The median time between intake and outcome was 6 days, while the mean was approximately 20.97 days. The difference suggests that a smaller number of animals had substantially longer periods between recorded events.

One limitation is that matching records by chronological sequence assumes that the nth intake corresponds to the nth outcome for the same animal. The shared animal ID and chronological ordering make this a reasonable approach for integration, but they do not independently confirm that every pair represents the same shelter episode. The results should therefore be interpreted with this limitation in mind.

### One Thing That Surprised Me

One thing that surprised me was that 139 matched intake and outcome event pairs had an outcome date earlier than the corresponding intake date. I initially found 780 negative time differences when comparing exact timestamps, but further investigation showed that many of these were caused by time-of-day or timezone differences on the same calendar date. After comparing calendar dates instead, I identified 139 pairs where the outcome genuinely appeared to precede the intake. I excluded those pairs from the analysis rather than treating negative time-to-outcome values as valid.



---


**AI Use Note:** I used ChatGPT to help understand the assignment requirements, check my reasoning, and develop and review some of my wording and code. I understood and reviewed the information it provided, including independently verifying important information and data access, and made sure I could explain and defend the decisions in this milestone myself.